# プロンプト

プロンプト「
```
以下の{# コード}を説明してください。
各過程を次の二通りの方法で詳しく説明してください。
1. 各過程のPythonコードの説明でなく，何を行っているのか文章で説明してください．可能ならば式で説明してください．
2. 各過程のPythonコードを何をやっているのかを説明してください．各過程で入力変数名、出力変数名を明記してください。

# コード
#!/usr/bin/env python
# coding: utf-8

# ### 閃亜鉛鉱構造とウルツ鉱構造の安定性の分類
# 
# **データ取得からデータ解析**
# 

# In[ ]:


from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')


get_ipython().run_line_magic('matplotlib', 'inline')
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)


# In[ ]:


# データ取得
def get_data():
    df = pd.read_csv("../data/ZB_WZ_dE_rawdescriptor.csv")
    descriptor_names = ['IP_A', 'EA_A', 'EN_A', 'Highest_occ_A',
                         'Lowest_unocc_A', 'rs_A', 'rp_A', 'rd_A', 'IP_B', 'EA_B', 'EN_B',
                         'Highest_occ_B', 'Lowest_unocc_B', 'rs_B', 'rp_B', 'rd_B']
    target_name = "dE"
    return df, descriptor_names, target_name

g_df, g_descriptor_names, g_target_name = get_data()


# 次の関数で分類問題用に
# ```python
# y = yraw >0 
# ```
# として目的変数を２クラスに変換しています。

# In[ ]:


def classify_df(df, descriptor_names, target_name):
    """データを分類する。

    Args:
        df (pd.DataFrame): データ
        descriptor_names ([str]): 説明変数名リスト
        target_name ([str])): 目的変数名リスト

    Returns:
        LogisticRegressionCV: LogisticRegressionCVインスタンス .
        np.ndarray: X.
        np.ndarray: y.
        np.ndarray: values of cls.predict(X)
        np.ndarray: values of cls.predict_proba(X)
    """
    Xraw = df.loc[:, descriptor_names].values
    yraw = df.loc[:, target_name].values
    y = yraw > 0

    # データプリプロセス
    scaler = StandardScaler()
    scaler.fit(Xraw)
    X = scaler.transform(Xraw)

    # データ解析
    kf = KFold(5, shuffle=True)
    cls = LogisticRegressionCV(cv=kf)
    cls.fit(X, y)
    score = cls.score(X, y)
    print("score=", score)
    yp = cls.predict(X)
    yp_proba = cls.predict_proba(X)
    print(classification_report(y, yp))
    index = []
    columns= []
    for s in cls.classes_:
        index.append("actual({})".format(s))
        columns.append("predict({})".format(s))
    cmdf = pd.DataFrame(confusion_matrix(y, yp), index=index, columns=columns)
    display(cmdf)
    
    return cls, X, y, yp, yp_proba

g_cls, g_X, g_y, g_yp, g_yp_proba =  classify_df(g_df, g_descriptor_names, g_target_name)


# **可視化**

# In[ ]:


g_df


# In[ ]:


def plot_X(X):
    """説明変数の図示。

    Args:
        X (np.ndarray): 説明変数
    """
    fig, ax = plt.subplots()
    ax.plot(X)
    ax.set_xlabel("index")
    ax.set_ylabel("X")
    
plot_X(g_X)

from collections import Counter
print("Counter", Counter(g_y))


# In[ ]:


g_Copt = g_cls.C_[0]
print("Copt=", g_Copt)


# 二値の場合はTrueに対して値が入っています。
# 
# CVの可視化を行います。

# In[ ]:


def plot_CV_scores(cls):
    """cls.scoreの表示。

    Args:
        cls (LogisticRegressionCV): LogisticRegressionCVインスタンス.

    """
    scores_mean = np.mean(cls.scores_[True], axis=0)
    scores_std = np.std(cls.scores_[True], axis=0)
    ic = np.argmax(scores_mean)
    print("index=", ic, "score=", scores_mean[ic])

    fig, ax = plt.subplots()
    ax.errorbar(np.log10(cls.Cs_), scores_mean, yerr=scores_std, capsize=5)
    ax.set_xlabel("log10(C)")
    ax.set_ylabel("score")
    
plot_CV_scores(g_cls)


# ### 付録
# boxplotでも見ておきます。

# In[ ]:


def plot_CV_scores_as_boxplot(cls, save_fig: bool=False):
    """cls.scoreの表示をboxplotで行う。

    Args:
        cls (LogisticRegressionCV): LogisticRegressionCVインスタンス.

    """    
    labels = []
    for x in np.log10(cls.Cs_):
        labels.append("{:.3f}".format(x))
    df_score = pd.DataFrame(cls.scores_[True], columns=labels)
    fig, ax = plt.subplots()
    df_score.boxplot(rot=90, ax=ax)
    ax.set_xlabel("log10(C)")
    if save_fig:
        fig.savefig("image_executed/ZB_WZ_cls_boxplot.png")
    
plot_CV_scores_as_boxplot(g_cls)


# ### CV(test)の表示
# 
# LogisticRegressionCVは観測データ全てを用いた評価値なので、
# CV(test)だけを表示する。
# 

# In[ ]:


def calc_CV_score(X,y, Copt):
    """CVを行い、score、y, ypを出力する。

    Args:
        X (np.ndarray)): 説明変数
        y (np.ndarray): 目的変数
        Copt (float): C of logistic regression.
        
    Returns:
        [float]: a list of KFold scores.
        [np.array]: a list of KFold y_test.
        [np.array]: a list of KFold predicted y_test.

    """
    ytest_list = []
    ytestp_list = []
    score_list = []
    kf = KFold(5, shuffle=True)
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        cls = LogisticRegression(C=Copt)
        cls.fit(Xtrain, ytrain)
        ytestp = cls.predict(Xtest)
        ytest_list.extend(ytest)
        ytestp_list.extend(ytestp)
        score = cls.score(Xtest, ytest)
        score_list.append(score)
    return score_list, ytest_list, ytestp_list

g_score_list, g_ytest_list, g_ytestp_list = calc_CV_score(g_X, g_y, g_Copt)


# In[ ]:


def show_scores(score_list, ytest_list, ytestp_list, classes):
    """score, y, ypの図示。

    Args:
        score_list ([float]]): a list of KFold scores.
        ytest_list ([np.array]): a list of y_test
        ytestp_list ([np.array]): a list of predicted y_test
        classes ([int]): classification classes.
    """
    print("score = {}({})".format(np.mean(score_list), np.std(score_list)))
    print(classification_report(ytest_list, ytestp_list))
    with open("image_executed/ZB_WZ_cls_report.txt", "w") as f:
        f.write((classification_report(ytest_list, ytestp_list)))
    index = []
    columns= []
    for s in classes:
        index.append("actual({})".format(s))
        columns.append("predict({})".format(s))
    df_cm = pd.DataFrame(confusion_matrix(ytest_list, ytestp_list), index=index,
                         columns=columns)
    display(df_cm)
    
show_scores(g_score_list, g_ytest_list, g_ytestp_list, g_cls.classes_ )


# **参考文献**
# 
# 1. Luca M. Ghiringhelli,
# Jan Vybiral,
# Sergey V. Levchenko,
# Claudia Draxl,
# and Matthias Scheffler,
# "Big Data of Materials Science: Critical Role of the Descriptor",
# Phys. Rev. Lett. 114, 105503 (2015)
# 

# ## 問題
# 
# ../data_calculated/ZB_WZ_dE_3var.csvを使って同じことを行え。

# In[ ]:

```
」

## 回答

以下に示すプログラムは、閃亜鉛鉱構造とウルツ鉱構造の安定性を分類するためのモデルを構築し、評価する一連の過程を含んでいます。これを二通りの方法で詳しく説明していきます。

1. **過程の内容を文章で説明**

   - **ライブラリのインポートと基本設定**:
     - 機械学習、データ解析、可視化のためのライブラリ（`scikit-learn`, `pandas`, `matplotlib`など）をインポートします。また、データフレームの表示設定と警告の無視設定も行います。
   
   - **データ取得**:
     - `get_data()`関数を用いて、CSVファイルからデータを読み込みます。このデータには様々な記述子（属性）とターゲット変数`dE`が含まれています。
   
   - **データの前処理と分類**:
     - `classify_df()`関数でデータを前処理し、ロジスティック回帰による分類を行います。ターゲット変数`dE`が0より大きいかで二クラスに変換します。また、標準化を行い、交差検証を用いてモデルを学習・評価します。結果として予測スコアや分類レポート、混同行列が出力されます。
   
   - **データの可視化**:
     - `plot_X()`を用いて説明変数をプロットします。
     - `plot_CV_scores()`を用いてパラメータCに対する交差検証スコアを可視化します。
     - `plot_CV_scores_as_boxplot()`を用いてスコアをボックスプロットで可視化します。
   
   - **モデル評価**:
     - 最適なCを用いたモデル評価を行い、交差検証のスコア、実際のクラスと予測クラスを出力します。
     - 結果をもとに、別のデータセットに対する評価を促しています。

2. **Pythonコードの詳細な説明**

   - **ライブラリのインポートと基本設定**:
     - 機械学習のために`LogisticRegressionCV`と`LogisticRegression`を、可視化には`matplotlib.pyplot`、データ操作には`pandas`をインポートしています。警告を無視し、データフレームの表示行数と列数を設定しています。

   - **データ取得**:
     - `get_data()`関数では、`pandas`を用いてCSVファイルを読み込み、特定の記述子とターゲットを取得してそれぞれ`descriptor_names`と`target_name`として返しています。

   - **データの前処理と分類** (`classify_df`関数):
     - `df`から`descriptor_names`に基づいて説明変数`Xraw`を取り出し、目的変数`yraw`は`target_name`から取得、その後`y = yraw > 0`で二値に変換します。
     - `StandardScaler`を用いて`Xraw`を標準化し、`KFold`による5分割交差検証で`LogisticRegressionCV`を行い、モデルの`cls`を学習させています。得られたスコア、予測値、予測確率を出力します。

   - **データの可視化**:
     - `plot_X()`で説明変数`X`をプロット。`Counter`を用いてクラスの頻度も出力します。
     - `plot_CV_scores()`で交差検証スコアをプロット。`plot_CV_scores_as_boxplot()`でスコアをボックスプロットしています。

   - **モデル評価** (`calc_CV_score`, `show_scores`):
     - 最適なCを用いて、新たに`LogisticRegression`を学習し、複数の交差検証スコアを計算し、最終的な分類結果を混同行列とともに出力します。

これらのステップにより、構造の安定性分類を試み、その結果を様々な方法で可視化及び評価しています。

## プロンプト

コード生成を依頼する．

プロンプト「
```
# 依頼
{## 過程}に従いPythonコードを書いてください．

## 過程

1. 観測データg_df, 説明変数g_descriptor_names，目的変数g_descriptor_namesを取得する．
```
g_df, g_descriptor_names, g_target_name = get_data()
```
2. 説明変数をg_dfのg_descriptor_names変数カラムから取得する．目的変数をg_dfのg_target_name変数カラムから取得する．
3. 目的変数を正の値か，それ以外かでカテゴリ値に変換する．
4. 説明変数を規格化する．
5. logistic regressionモデルを用いる．
6. ５回交差検定でlogistic regressionモデルのハイパーパラメタを求める．
7. 全データを用いてlogistic regressionモデルを学習しなおす．
8. 混同行列，accuracy, precision, recall, f1 scoreを出力してください．
```
」

In [ ]:
# データ取得
import pandas as pd
def get_data():
    df = pd.read_csv("../data/ZB_WZ_dE_rawdescriptor.csv")
    descriptor_names = ['IP_A', 'EA_A', 'EN_A', 'Highest_occ_A',
                         'Lowest_unocc_A', 'rs_A', 'rp_A', 'rd_A', 'IP_B', 'EA_B', 'EN_B',
                         'Highest_occ_B', 'Lowest_unocc_B', 'rs_B', 'rp_B', 'rd_B']
    target_name = "dE"
    return df, descriptor_names, target_name

g_df, g_descriptor_names, g_target_name = get_data()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. 観測データg_df, 説明変数g_descriptor_names，目的変数g_target_nameを取得する．

# 2. 説明変数をg_dfのg_descriptor_names変数カラムから取得する．
# 目的変数をg_dfのg_target_name変数カラムから取得する．
X = g_df[g_descriptor_names]
y = g_df[g_target_name]

# 3. 目的変数を正の値か，それ以外かでカテゴリ値に変換する．
y = np.where(y > 0, 1, 0)

# 4. 説明変数を規格化する．
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. logistic regressionモデルを用いる．
log_reg = LogisticRegression()

# 6. ５回交差検定でlogistic regressionモデルのハイパーパラメタを求める．
param_grid = {
    'C': [0.1, 1, 10, 100],
    'solver': ['liblinear', 'saga']
}
grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_scaled, y)

# 最適ハイパーパラメータ
best_params = grid_search.best_params_
print("Best Parameters:", best_params)

# 7. 全データを用いてlogistic regressionモデルを学習しなおす．
best_log_reg = LogisticRegression(**best_params)
best_log_reg.fit(X_scaled, y)

# 8. 混同行列，accuracy, precision, recall, f1 scoreを出力してください．
y_pred = best_log_reg.predict(X_scaled)

conf_matrix = confusion_matrix(y, y_pred)
accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)

print("Confusion Matrix:\n", conf_matrix)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


## プロンプト

プロンプト「
ハイパーパラメタに対するaccuracy scoreを図示してください。」


## 回答

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. 観測データg_df, 説明変数g_descriptor_names，目的変数g_target_nameを取得する．
g_df, g_descriptor_names, g_target_name = get_data()

# 2. 説明変数をg_dfのg_descriptor_names変数カラムから取得する．
# 目的変数をg_dfのg_target_name変数カラムから取得する．
X = g_df[g_descriptor_names]
y = g_df[g_target_name]

# 3. 目的変数を正の値か，それ以外かでカテゴリ値に変換する．
y = np.where(y > 0, 1, 0)

# 4. 説明変数を規格化する．
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. logistic regressionモデルを用いる．
log_reg = LogisticRegression()

# 6. ５回交差検定でlogistic regressionモデルのハイパーパラメタを求める．
param_grid = {
    'C': np.logspace(-3,7, 30), # 修正
    'solver': ['liblinear', 'saga']
}
grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_scaled, y)

# 最適ハイパーパラメータ
best_params = grid_search.best_params_
print("Best Parameters:", best_params)

# 精度スコアを取得
mean_test_scores = grid_search.cv_results_['mean_test_score']
param_combinations = grid_search.cv_results_['params']

# 7. ハイパーパラメタに対するaccuracy scoreを図示
plt.figure(figsize=(10, 6))
for solver in ['liblinear', 'saga']:
    scores = [mean_test_scores[i] for i in range(len(param_combinations)) if param_combinations[i]['solver'] == solver]
    C_values = [param_combinations[i]['C'] for i in range(len(param_combinations)) if param_combinations[i]['solver'] == solver]
    plt.plot(C_values, scores, marker='o', label=f'solver={solver}')
    
plt.xlabel('C value')
plt.ylabel('Mean Accuracy Score')
plt.title('Accuracy Score vs. C for different solvers')
plt.xscale('log')
plt.legend()
plt.grid(True)
plt.show()
